In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax import vmap
from src.fdm import ECirreMechanismFDMSolver, EMechanismFDMSolver
from src.params import ECirreMechanismFDMParams, EMechanismFDMParams
from src.voltammetry import LinearSweepDC

plt.style.use("seaborn-v0_8-darkgrid")

# Current Visualisation


## E Reaction Mechanism


In [ ]:
voltammetry = LinearSweepDC()
fdm_solver = EMechanismFDMSolver(voltammetry, 1e-2, 5e-2)

base_params = EMechanismFDMParams(
    alpha=jnp.array(0.7), K0=jnp.array(10.0), E0=jnp.array(0.0), dB=jnp.array(1.0)
)

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))

# Alpha Varying

alpha_range = jnp.linspace(0.3, 0.7, 5)
K0_params = EMechanismFDMParams(
    alpha=alpha_range,
    K0=jnp.full_like(alpha_range, base_params.K0),
    E0=jnp.full_like(alpha_range, base_params.E0),
    dB=jnp.full_like(alpha_range, base_params.dB),
)

currents = vmap(fdm_solver.solve)(K0_params)

for val, current in zip(alpha_range, currents):
    ax[0, 0].plot(fdm_solver.applied_potentials, current, label=val)
ax[0, 0].xaxis.set_inverted(True)
ax[0, 0].yaxis.set_inverted(True)
ax[0, 0].legend()

# K0 Varying
K0_range = jnp.power(10.0, jnp.linspace(0.0, 5, 6))
K0_params = EMechanismFDMParams(
    alpha=jnp.full_like(K0_range, base_params.alpha),
    E0=jnp.full_like(K0_range, base_params.E0),
    dB=jnp.full_like(K0_range, base_params.dB),
    K0=K0_range,
)

currents = vmap(fdm_solver.solve)(K0_params)

for val, current in zip(K0_range, currents):
    ax[0, 1].plot(fdm_solver.applied_potentials, current, label=f"{val:.0e}")
ax[0, 1].xaxis.set_inverted(True)
ax[0, 1].yaxis.set_inverted(True)
ax[0, 1].legend()

# E0 Varying
E0_range = jnp.linspace(-2.0, 2.0, 5)
E0_params = EMechanismFDMParams(
    alpha=jnp.full_like(E0_range, base_params.alpha),
    E0=E0_range,
    dB=jnp.full_like(E0_range, base_params.dB),
    K0=jnp.full_like(E0_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(E0_params)

for val, current in zip(E0_range, currents):
    ax[1, 0].plot(fdm_solver.applied_potentials, current, label=val)

ax[1, 0].xaxis.set_inverted(True)
ax[1, 0].yaxis.set_inverted(True)
ax[1, 0].legend()

# dB Varying
dB_range = jnp.array([0.5, 0.8, 1.0, 2.0, 3.0])

dB_params = EMechanismFDMParams(
    alpha=jnp.full_like(dB_range, base_params.alpha),
    E0=jnp.full_like(dB_range, base_params.E0),
    dB=dB_range,
    K0=jnp.full_like(dB_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(dB_params)

for val, current in zip(dB_range, currents):
    ax[1, 1].plot(fdm_solver.applied_potentials, current, label=val)

ax[1, 1].xaxis.set_inverted(True)
ax[1, 1].yaxis.set_inverted(True)
ax[1, 1].legend()

plt.show()


## EC_irre Current


In [ ]:
voltammetry = LinearSweepDC()

fdm_solver = ECirreMechanismFDMSolver(voltammetry)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

base_params = ECirreMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(10.0),
    Kminus=jnp.array(5.0),
    Kplus=jnp.array(10.0),
    E0=jnp.array(0.0),
    dB=jnp.array(1.0),
)

Kplus_range = jnp.linspace(1.0, 20.0, 5)

Kplus_params = ECirreMechanismFDMParams(
    alpha=jnp.full_like(Kplus_range, base_params.alpha),
    K0=jnp.full_like(Kplus_range, base_params.K0),
    Kminus=jnp.full_like(Kplus_range, base_params.Kminus),
    Kplus=Kplus_range,
    E0=jnp.full_like(Kplus_range, base_params.E0),
    dB=jnp.full_like(Kplus_range, base_params.dB),
)


currents = vmap(fdm_solver.solve)(Kplus_params)
for current, kplus in zip(currents, Kplus_range):
    ax1.plot(fdm_solver.applied_potentials, current, label=kplus)

ax1.legend()
ax1.xaxis.set_inverted(True)
ax1.yaxis.set_inverted(True)

Kminus_range = jnp.linspace(1.0, 20.0, 5)

Kminus_params = ECirreMechanismFDMParams(
    alpha=jnp.full_like(Kminus_range, base_params.alpha),
    K0=jnp.full_like(Kminus_range, base_params.K0),
    Kminus=Kminus_range,
    Kplus=jnp.full_like(Kminus_range, base_params.Kplus),
    E0=jnp.full_like(Kminus_range, base_params.E0),
    dB=jnp.full_like(Kminus_range, base_params.dB),
)


currents = vmap(fdm_solver.solve)(Kminus_params)
for current, kminus in zip(currents, Kplus_range):
    ax2.plot(fdm_solver.applied_potentials, current, label=kminus)

ax2.legend()
ax2.xaxis.set_inverted(True)
ax2.yaxis.set_inverted(True)
plt.tight_layout()
plt.show()

# Sampling


## E Reaction Mechanism


### Plotting Function

In [ ]:
def plot_e_datasets(datasets):
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(ncols=4, figsize=(20, 5))

    true_params = EMechanismFDMParams(
        alpha=jnp.array(0.6),
        K0=jnp.array(10.0),
        E0=jnp.array(2.0),
        dB=jnp.array(0.5),
    )

    hist_kwargs = dict(
        alpha=0.75,
        bins=50,
        density=True,
    )

    for d in datasets.keys():
        ax1.hist(datasets[d]["alpha"], label=d, **hist_kwargs)
        ax2.hist(datasets[d]["K0"], **hist_kwargs)
        ax3.hist(datasets[d]["E0"], **hist_kwargs)
        ax4.hist(datasets[d]["dB"], **hist_kwargs)

    ax1.set_title("alpha")
    ax1.axvline(true_params.alpha, c="black", linestyle="--")
    ax2.set_title("K0")
    ax2.axvline(true_params.K0, c="black", linestyle="--")
    ax3.set_title("E0")
    ax3.axvline(true_params.E0, c="black", linestyle="--")
    ax4.set_title("dB")
    ax4.axvline(true_params.dB, c="black", linestyle="--")

    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
    )

    fig.suptitle(
        "Posterior distributions obtained using Linear Sweep DC Voltammetry",
        fontsize=18,
        y=0.98,
    )

    plt.show()

    print(f"{'Method':<8} {'Param':<10} {'μ ± σ':<30}")
    print("-" * 52)

    for method, params in datasets.items():
        for param, values in params.items():
            mu = np.mean(values)
            sigma = np.std(values)
            summary = f"{mu:.4e} ± {sigma:.4e}"
            print(f"{method:<8} {param:<10} {summary:<30}")
        print("-" * 28)
    print()


### Data

In [ ]:
rw = np.load("./data/E_MetropolisHastings_LinearSweepDC.npz")
pathfinder = np.load("./data/E_Pathfinder_LinearSweepDC.npz")
nuts = np.load("./data/E_Nuts_LinearSweepDC.npz")

datasets = {
    "RW": {
        "alpha": rw["alpha"][10_000:],
        "K0": rw["K0"][10_000:],
        "E0": rw["E0"][10_000:],
        "dB": rw["dB"][10_000:],
    },
    # "VI": {
    #     "alpha": pathfinder["alpha"],
    #     "K0": pathfinder["K0"],
    #     "E0": pathfinder["E0"],
    #     "dB": pathfinder["dB"],
    # },
    "NUTS": {
        "alpha": nuts["alpha"],
        "K0": nuts["K0"],
        "E0": nuts["E0"],
        "dB": nuts["dB"],
    },
}

plot_e_datasets(datasets)

In [ ]:
nuts = np.load("./data/E_Nuts_LinearSweepAC.npz")
rw = np.load("./data/E_MetropolisHastings_LinearSweepAC.npz")
pathfinder = np.load("./data/E_Pathfinder_LinearSweepAC.npz")

datasets = {
    "RW": {
        "alpha": rw["alpha"][10_000:],
        "K0": rw["K0"][10_000:],
        "E0": rw["E0"][10_000:],
        "dB": rw["dB"][10_000:],
    },
    # "VI": {
    #     "alpha": pathfinder["alpha"],
    #     "K0": pathfinder["K0"],
    #     "E0": pathfinder["E0"],
    #     "dB": pathfinder["dB"],
    # },
    "NUTS": {
        "alpha": nuts["alpha"],
        "K0": nuts["K0"],
        "E0": nuts["E0"],
        "dB": nuts["dB"],
    },
}

plot_e_datasets(datasets)

## ECirre Reaction


### Plotting Function

In [ ]:
def plot_ec_irre_datasets(datasets):
    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(20, 5))

    true_params = ECirreMechanismFDMParams(
        alpha=jnp.array(0.6),
        K0=jnp.array(10.0),
        Kplus=jnp.array(10.0),
        Kminus=jnp.array(10.0),
        E0=jnp.array(2.0),
        dB=jnp.array(0.5),
    )

    hist_kwargs = dict(
        alpha=0.75,
        bins=50,
        density=True,
    )

    for d in datasets.keys():
        axs[0, 0].hist(datasets[d]["alpha"], label=d, **hist_kwargs)
        axs[0, 1].hist(datasets[d]["K0"], **hist_kwargs)
        axs[0, 2].hist(datasets[d]["E0"], **hist_kwargs)
        axs[1, 0].hist(datasets[d]["dB"], **hist_kwargs)
        axs[1, 1].hist(datasets[d]["Kplus"], **hist_kwargs)
        axs[1, 2].hist(datasets[d]["Kminus"], **hist_kwargs)

    axs[0, 0].set_title(r"$\alpha$")
    axs[0, 0].axvline(true_params.alpha, c="black", linestyle="--")
    axs[0, 1].set_title(r"$K_0$")
    axs[0, 1].axvline(true_params.K0, c="black", linestyle="--")
    axs[0, 2].set_title(r"$E_0$")
    axs[0, 2].axvline(true_params.E0, c="black", linestyle="--")

    axs[1, 0].set_title(r"$d_B$")
    axs[1, 0].axvline(true_params.dB, c="black", linestyle="--")
    axs[1, 1].set_title(r"$K_+$")
    axs[1, 1].axvline(true_params.Kplus, c="black", linestyle="--")
    axs[1, 2].set_title(r"$K_-$")
    axs[1, 2].axvline(true_params.Kminus, c="black", linestyle="--")

    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper left",
        ncol=3,
        frameon=False,
    )

    fig.suptitle(
        "Posterior distributions obtained using Linear Sweep DC Voltammetry",
        fontsize=18,
        y=0.98,
    )

    plt.tight_layout()
    plt.show()

    print(f"{'Method':<8} {'Param':<10} {'μ ± σ':<30}")
    print("-" * 52)

    for method, params in datasets.items():
        for param, values in params.items():
            mu = np.mean(values)
            sigma = np.std(values)
            summary = f"{mu:.4e} ± {sigma:.4e}"
            print(f"{method:<8} {param:<10} {summary:<30}")
        print("-" * 28)
    print()

### Data

In [ ]:
rw = np.load("./data/ECirre_MetropolisHastings_LinearSweepDC.npz")
pathfinder = np.load("./data/ECirre_Pathfinder_LinearSweepDC.npz")
nuts = np.load("./data/ECirre_Nuts_LinearSweepDC.npz")


datasets = {
    "RW": {
        "alpha": rw["alpha"][10_000:],
        "K0": rw["K0"][10_000:],
        "Kplus": rw["Kplus"][10_000:],
        "Kminus": rw["Kminus"][10_000:],
        "E0": rw["E0"][10_000:],
        "dB": rw["dB"][10_000:],
    },
    # "VI": {
    #     "alpha": pathfinder["alpha"],
    #     "K0": pathfinder["K0"],
    #     "Kplus": pathfinder["Kplus"],
    #     "Kminus": pathfinder["Kminus"],
    #     "E0": pathfinder["E0"],
    #     "dB": pathfinder["dB"],
    # },
    "NUTS": {
        "alpha": nuts["alpha"],
        "K0": nuts["K0"],
        "Kplus": nuts["Kplus"],
        "Kminus": nuts["Kminus"],
        "E0": nuts["E0"],
        "dB": nuts["dB"],
    },
}

plot_ec_irre_datasets(datasets)


In [ ]:
rw = np.load("./data/ECirre_MetropolisHastings_LinearSweepAC.npz")
pathfinder = np.load("./data/ECirre_Pathfinder_LinearSweepAC.npz")
nuts = np.load("./data/ECirre_Nuts_LinearSweepAC.npz")


datasets = {
    "RW": {
        "alpha": rw["alpha"][10_000:],
        "K0": rw["K0"][10_000:],
        "Kplus": rw["Kplus"][10_000:],
        "Kminus": rw["Kminus"][10_000:],
        "E0": rw["E0"][10_000:],
        "dB": rw["dB"][10_000:],
    },
    # "VI": {
    #     "alpha": pathfinder["alpha"],
    #     "K0": pathfinder["K0"],
    #     "Kplus": pathfinder["Kplus"],
    #     "Kminus": pathfinder["Kminus"],
    #     "E0": pathfinder["E0"],
    #     "dB": pathfinder["dB"],
    # },
    "NUTS": {
        "alpha": nuts["alpha"],
        "K0": nuts["K0"],
        "Kplus": nuts["Kplus"],
        "Kminus": nuts["Kminus"],
        "E0": nuts["E0"],
        "dB": nuts["dB"],
    },
}

plot_ec_irre_datasets(datasets)